In [1]:
%pip install python-chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 5.0 MB/s  0:00:01 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147879 sha256=a1f5a02199684fb5b5ea020757503453d127d59220338b7d8df837a96a4965d6
  Stored in directory: /Users/adamz/Library/Caches/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [python-chess]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import chess.pgn
import os

In [3]:
filename = 'raw_db.pgn'

try:
    with open(filename, "r", encoding="utf-8") as pgn_file:
        # Read the first game from the file
        game = chess.pgn.read_game(pgn_file)

        if game:
            print(f"--- Headers for: {filename} ---")
            
            # Display specific common headers
            print(f"Event: {game.headers.get('Event', '?')}")
            print(f"Date:  {game.headers.get('Date', '?')}")
            print(f"White: {game.headers.get('White', '?')}")
            print(f"Black: {game.headers.get('Black', '?')}")
            print(f"Result: {game.headers.get('Result', '*')}")
            
            print("\n--- All Raw Headers ---")
            # Iterate through all headers found in the game
            for key, value in game.headers.items():
                print(f'[{key} "{value}"]')
        else:
            print("No games found in the file.")
            

except FileNotFoundError:
    print(f"Error: The file '{filename}' was not found.")

--- Headers for: raw_db.pgn ---
Event: Rated Classical game
Date:  ????.??.??
White: BFG9k
Black: mamalak
Result: 1-0

--- All Raw Headers ---
[Event "Rated Classical game"]
[Site "https://lichess.org/j1dkb5dw"]
[Date "????.??.??"]
[Round "?"]
[White "BFG9k"]
[Black "mamalak"]
[Result "1-0"]
[UTCDate "2012.12.31"]
[UTCTime "23:01:03"]
[WhiteElo "1639"]
[BlackElo "1403"]
[WhiteRatingDiff "+5"]
[BlackRatingDiff "-8"]
[ECO "C00"]
[Opening "French Defense: Normal Variation"]
[TimeControl "600+8"]
[Termination "Normal"]


In [14]:
from pathlib import Path
import os

def get_safe_elo(headers, key):
    """Attempts to parse Elo, returns None if '?' or missing."""
    val = headers.get(key)
    if val is None or val == "?":
        return None
    try:
        return int(val)
    except ValueError:
        return None

def separate_games_by_elo(input_pgn, output_dir, compartments):
    compartments = sorted(compartments)
    compartment_count = {}
    for split in compartments:
        compartment_count[split] = 0
    
    os.mkdir(output_dir)

    i = 0
    with open(input_pgn, "r") as pgn_file:
        while True:
            i += 1
            if i % 10000 == 0:
                print(f"iter: {i}")
                for compartment in compartments:
                    print(f"\t{compartment}: {compartment_count[compartment]}")
            
            
            game = chess.pgn.read_game(pgn_file)
            if game is None:
                break 

            # Safely extract ratings
            w_elo = get_safe_elo(game.headers, "WhiteElo")
            b_elo = get_safe_elo(game.headers, "BlackElo")

            if w_elo is None or b_elo is None:
                continue

            mean_elo = (w_elo + b_elo) / 2

            # Logic to find the bucket
            target_threshold = 0
            for limit in compartments:
                if mean_elo >= limit:
                    target_threshold = limit
                else:
                    break
            
            compartment_count[target_threshold] += 1
            filename = f"games_elo_{target_threshold}.pgn"
            with open(output_dir / filename, "a") as out_pgn:
                out_pgn.write(str(game) + "\n\n")
                
    
    print("finish data")
    for compartment in compartments:
        print(f"\t{compartment}: {compartment_count[compartment]}")

COMPARTMENTS = [800, 1000, 1200, 1400, 1600, 1800, 2000]
separate_games_by_elo(filename, Path('./split_pgn'), COMPARTMENTS)

iter: 10000
	800: 1
	1000: 68
	1200: 1102
	1400: 4096
	1600: 3326
	1800: 1270
	2000: 115
iter: 20000
	800: 1
	1000: 155
	1200: 2326
	1400: 8212
	1600: 6625
	1800: 2330
	2000: 310
iter: 30000
	800: 1
	1000: 247
	1200: 3355
	1400: 12182
	1600: 10326
	1800: 3393
	2000: 430
iter: 40000
	800: 1
	1000: 308
	1200: 4384
	1400: 16248
	1600: 13814
	1800: 4502
	2000: 650


KeyboardInterrupt: 